# 🌳 Notebook 04: Backtracking Iterative Quantum Branch-and-Bound (`IQO-BnB`) & `MPS` Tensor Networks
**PravahaQ (`v0.1.0`) — Overcoming Frustrated Spin Traps & Simulating Large Systems**

---

## 🟢 Layer 1: Plain-English Intuition

1. **Why Greedy Recursive QAOA (RQAOA) Fails:** Standard RQAOA measures the strongest pair of spins $(i^*, j^*)$ and permanently glues them together ($Z_{j^*} = \pm Z_{i^*}$). On a frustrated 3-body graph, making one greedy mistake on step 1 locks out the true optimum forever.
2. **Our Fix (`IQO-BnB`):** We build a **Backtracking Branch-and-Bound Tree** that tests both parities ($+1$ and $-1$) on high-frustration splits and algebraically folds **3-body cubic terms into 1-body linear fields**:
   $$K_{i,j,k} \hat{Z}_i (\sigma \hat{Z}_i) \hat{Z}_k = \sigma K_{i,j,k} \hat{Z}_k \quad (\text{since } \hat{Z}_i^2 = \hat{I})$$
3. **Matrix Product State (`MPS`) Tensor Network Surrogate:** Represents shallow counterdiabatic states as a 1D chain of SVD-truncated tensors ($\chi_{\max}$), pre-screening circuits in milliseconds.

In [1]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent / "src"))
sys.path.insert(0, str(Path.cwd() / "src"))

from pravaha_q.domain import generate_coupled_problem
from pravaha_q.math_engine import VaisheshikaHUBOCompiler, hubo_to_ising
from pravaha_q.algorithms import IQOBranchAndBoundSolver
from pravaha_q.tensor_network import MPSSimulator

prob = generate_coupled_problem(num_nodes=2, num_tasks=3, num_power_states=1, seed=42)
hubo = VaisheshikaHUBOCompiler().compile(prob)
ising = hubo_to_ising(hubo)

iqo_solver = IQOBranchAndBoundSolver(classical_cutoff_vars=4, max_backtrack_depth=2, seed=42)
iqo_res = iqo_solver.solve(ising)

print("=== Backtracking IQO-BnB Variable Reduction Trace ===")
print(f"Branches Explored          : {iqo_res.branches_explored}")
print(f"Greedy Path Energy         : {iqo_res.greedy_rqaoa_energy:.4f}")
print(f"IQO-BnB Best Energy        : {iqo_res.best_energy:.4f} | Best Bitstring: |{iqo_res.best_bitstring}>\n")

print("Step | Remaining Qubits | Eliminated -> Reference | Parity | |<Z_i Z_j>| | Branch Type")
print("-" * 87)
for idx, st in enumerate(iqo_res.reduction_trace, start=1):
    btype = "Backtrack" if st.is_backtrack_branch else "Primary"
    print(
        f"  {idx}  |        {st.remaining_qubits}         |     "
        f"Q_{st.eliminated_qubit}   ->    Q_{st.reference_qubit}     |   {st.parity_sign:+d}   |   "
        f"{st.correlation_magnitude:.5f}   | {btype}"
    )

=== Backtracking IQO-BnB Variable Reduction Trace ===
Branches Explored          : 4
Greedy Path Energy         : -23.1480
IQO-BnB Best Energy        : -23.1480 | Best Bitstring: |01100101>

Step | Remaining Qubits | Eliminated -> Reference | Parity | |<Z_i Z_j>| | Branch Type
---------------------------------------------------------------------------------------
  1  |        7         |     Q_1   ->    Q_0     |   -1   |   0.74120   | Primary
  2  |        6         |     Q_3   ->    Q_2     |   -1   |   0.69840   | Primary
  3  |        5         |     Q_5   ->    Q_4     |   -1   |   0.65110   | Primary
  4  |        4         |     Q_6   ->    Q_0     |   +1   |   0.58930   | Primary


In [2]:
mps = MPSSimulator(max_bond_dim=32, seed=42)
mps_res = mps.solve(ising)
print("=== MPS Tensor Network SVD Surrogate Verification ===")
print(f"Max Bond Dimension (chi_max): {mps_res.max_bond_dimension}")
print(f"MPS Best Bitstring          : |{mps_res.best_bitstring}> | Energy: {mps_res.best_energy:.4f}")

=== MPS Tensor Network SVD Surrogate Verification ===
Max Bond Dimension (chi_max): 32
MPS Best Bitstring          : |01100101> | Energy: -23.1480
